# 02 ・ TMDB：正規 API 與比對的難題

## 這一章要做什麼

拿上一章的片名清單，去 TMDB（The Movie Database）換回完整的電影資料 ——
海報、評分、類型、簡介、上映日期。

## 和上一章的對比

| | 影城 API（01） | TMDB（02） |
|---|---|---|
| 文件 | 沒有 | 完整 |
| 金鑰 | 不用 | 要 |
| 速率限制 | 不明 | 有 |
| 回應結構 | 未知，要逆向 | 穩定，有文件 |
| 主要難題 | 資料在哪、資料很髒 | **查到的是不是同一部片** |

正規 API 不代表沒有坑。這一章的坑在**比對**：
你拿「藍色監獄」去搜尋，回來十筆結果，哪一筆才是你要的？

## 本章產出

`movieapp/tmdb.py`：

```python
tmdb.search(query)      # -> (搜尋結果, 錯誤)
tmdb.genres()           # -> ({類型id: 名稱}, 錯誤)
tmdb.best_match(q, res) # -> 最像的那一筆，或 None
tmdb.enrich(titles)     # -> ([{title, meta, source}, ...], 錯誤)
```

---
## 0 ・ 開場

這一章要用到上一章的產出，所以 `setup()` 多帶了 `requires` 參數。
如果你跳過了 01，這裡會直接告訴你該回去跑哪一本。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")
from movieapp.config import setup; setup(requires=["sources"])

---
## 1 ・ 金鑰放在 header，不要放在網址

TMDB 用 Read Access Token 認證，放在 `Authorization` header：

```python
headers = {"Authorization": f"Bearer {token}"}
```

**不要放在網址參數裡。** 差別很實際：程式出錯時，例外訊息、
伺服器日誌、瀏覽器歷史都可能把網址記下來 —— 放在網址裡的金鑰就跟著外流。
放在 header 就不會。

金鑰本身由 `config` 統一管理，讀取順序是「環境變數 → `.env` → 當場輸入」。

In [ ]:
from movieapp import config

token = config.tmdb_token(required=False)
print("金鑰:", (token[:12] + "…") if token else "未設定")

---
## 2 ・ 第一次查詢

先用最單純的方式查一部片，看看回來什麼。

In [ ]:
from movieapp import http

SEARCH_URL = "https://api.themoviedb.org/3/search/movie"
headers = {"Authorization": f"Bearer {token}", "accept": "application/json"}

data, error = http.fetch_json(
    SEARCH_URL,
    params={"query": "蜘蛛人：重生日", "language": "zh-TW"},
    headers=headers,
)

if error:
    print("失敗：", error)
else:
    print("回應的 key:", list(data))
    print("結果筆數:", len(data["results"]))
    first = data["results"][0]
    print()
    print("第一筆的欄位:")
    for key in list(first)[:12]:
        print(f"   {key:20} {str(first[key])[:50]}")

每一筆結果都有 `id`（TMDB 的電影編號）、`title`、`release_date`、
`vote_average`（評分）、`popularity`（熱門度）、`genre_ids`（類型）、
`poster_path`（海報）。

**`id` 特別重要** —— 它是這部電影在 TMDB 的唯一編號，
第 3 章要用它判斷「兩家影城的這兩筆是不是同一部片」。

---
## 3 ・ 類型對照表

`genre_ids` 是一串數字，要另外查對照表才知道 28 是什麼。
這張表很少變，查一次就好。

In [ ]:
GENRE_URL = "https://api.themoviedb.org/3/genre/movie/list"

data, error = http.fetch_json(GENRE_URL, params={"language": "zh-TW"},
                              headers=headers)
print([g["name"] for g in data["genres"]] if not error else error)

看出問題了嗎？**參數給的是 `zh-TW`，回來的卻是簡體中文**
（「动作」「冒险」而不是「動作」「冒險」）。

這是 TMDB 資料端的限制，換成 `zh-Hant`、`zh-HK` 都一樣。
總共只有 19 個類型，最務實的做法就是自己補一張對照表 ——
而且要對照 **id** 而不是名稱，因為 id 是穩定的。

這種「API 給的資料不完全符合需求，自己補一小塊」的情況很常見，
重點是補的方式要能撐過對方改版。

---
## 4 ・ 核心陷阱：第一筆不一定是你要的

原始版本的做法是直接取 `results[0]`。看看會發生什麼事：

In [ ]:
for query in ["Blue Lock", "藍色監獄", "LOOK BACK", "Pa Pa Go"]:
    data, error = http.fetch_json(
        SEARCH_URL, params={"query": query, "language": "zh-TW"},
        headers=headers)
    if error:
        print(f"  {query!r:14} -> 查詢失敗（{error[:40]}）")
        continue
    results = data.get("results", [])
    top = results[0] if results else None
    print(f"  {query!r:14} {len(results):2} 筆 -> results[0] = "
          f"{(top or {}).get('title', '(無結果)')!r}")

`Blue Lock` 查到的是完全不相干的電影，`Pa Pa Go` 也是。
但用中文名「藍色監獄」查就正確。

**兩個結論：**

1. **中文片名的命中率高很多。** 這就是上一章選 `name`（中文）
   而不是 `nameAlternative`（英文）的原因 —— 實測差距很明顯。
2. **不能盲目相信 `results[0]`。** TMDB 的排序是它自己的相關度，
   不保證第一筆就是你要的。

第 2 點怎麼解決？自己算相似度，挑最像的那一筆；
如果連最像的都不夠像，**寧可回 `None` 也不要放一部無關的電影上去**。

In [ ]:
import difflib
import re


def normalize(text):
    """比對前先把標點、空白、大小寫的差異抹平。"""
    return re.sub(r"[\s　:：,，.。!！?？'\"’\-—－]", "", str(text or "")).lower()


def match_score(query, movie):
    """片名和某筆搜尋結果的相似度，0.0 ~ 1.0。"""
    q = normalize(query)
    title = normalize(movie.get("title"))
    original = normalize(movie.get("original_title"))
    if not q:
        return 0.0
    if q == title or q == original:          # 完全一樣
        return 1.0
    if q in title or title in q or q in original or original in q:   # 一方包含另一方
        return 0.9
    return max(                               # 都不是就算字元相似度
        difflib.SequenceMatcher(None, q, title).ratio(),
        difflib.SequenceMatcher(None, q, original).ratio(),
    )


# 拿「藍色監獄」的搜尋結果，看看每一筆的分數
data, _ = http.fetch_json(SEARCH_URL,
                          params={"query": "藍色監獄", "language": "zh-TW"},
                          headers=headers)
for movie in (data or {}).get("results", [])[:5]:
    print(f"  {match_score('藍色監獄', movie):.2f}  {movie.get('title')!r}"
          f"  (熱門度 {movie.get('popularity', 0):.0f})")

---
## 5 ・ 快取：同一個片名不要查兩次

同一部電影兩家影城都有、學員重跑 cell、網頁重新整理 ——
同一個查詢很容易被重複送出。

TMDB 有速率限制，而且**限制是看 IP 的**。一整間教室共用一個對外 IP，
30 個人同時跑 50 次查詢就是 1500 次請求瞬間打出去，很容易被限流。

所以加一層有效期一小時的快取：

In [ ]:
import time

_CACHE = {}
_CACHE_TTL = 3600


def cached_search(query, language="zh-TW"):
    key = f"{query.lower()}|{language}"
    hit = _CACHE.get(key)
    if hit and time.time() - hit[0] < _CACHE_TTL:
        return hit[1], True            # True 代表這次是從快取拿的
    data, error = http.fetch_json(SEARCH_URL,
                                  params={"query": query, "language": language},
                                  headers=headers)
    results = (data or {}).get("results", [])
    if not error:
        _CACHE[key] = (time.time(), results)
    return results, False


for _ in range(3):
    results, from_cache = cached_search("奧德賽")
    print(f"  拿到 {len(results)} 筆，來自快取：{from_cache}")

---
## 6 ・ 寫成模組

In [ ]:
%%writefile ../movieapp/tmdb.py
"""TMDB（The Movie Database）查詢。

跟 01 的影城 API 完全相反：這是一支正規 API ——
有公開文件、要金鑰、有速率限制、回應結構穩定。

金鑰用 Read Access Token，放在 Authorization header（不是網址參數），
所以就算程式出錯把網址印出來，也不會把金鑰一起印出去。

本檔案由 notebooks/02_TMDB.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

import difflib
import re
import time
from concurrent.futures import ThreadPoolExecutor

from movieapp import config
from movieapp.http import fetch_json

SEARCH_URL = "https://api.themoviedb.org/3/search/movie"
GENRE_URL = "https://api.themoviedb.org/3/genre/movie/list"

# 海報與劇照的圖片網址前綴（w92 小圖、w300 中圖、original 原始尺寸）
IMAGE_BASE = "https://image.tmdb.org/t/p/"

# 同一個片名在一次上課中會被查很多次，用簡單的 TTL 快取擋掉重複請求
_CACHE = {}
_CACHE_TTL = 3600  # 秒

# 判定「這筆結果算不算命中」的相似度門檻
MATCH_THRESHOLD = 0.6

# TMDB 的 zh-TW 類型名稱實際回的是簡體中文（zh-Hant、zh-HK 也一樣），
# 這是它資料端的限制，改參數沒用。總共只有 19 個，直接補一張正體對照表。
# 對照的是類型 id 而不是名稱，因為 id 是穩定的。
ZH_TW_GENRES = {
    28: "動作", 12: "冒險", 16: "動畫", 35: "喜劇", 80: "犯罪",
    99: "紀錄", 18: "劇情", 10751: "家庭", 14: "奇幻", 36: "歷史",
    27: "恐怖", 10402: "音樂", 9648: "懸疑", 10749: "愛情", 878: "科幻",
    10770: "電視電影", 53: "驚悚", 10752: "戰爭", 37: "西部",
}


def _headers():
    return {
        "Authorization": f"Bearer {config.tmdb_token()}",
        "accept": "application/json",
    }


def clear_cache():
    _CACHE.clear()


def search(query, language="zh-TW", use_cache=True):
    """用片名查 TMDB。回傳 (results, error)。"""
    query = str(query or "").strip()
    if not query:
        return [], "缺少查詢字串"

    cache_key = f"{query.lower()}|{language}"
    if use_cache:
        hit = _CACHE.get(cache_key)
        if hit and time.time() - hit[0] < _CACHE_TTL:
            return hit[1], None

    data, error = fetch_json(
        SEARCH_URL,
        params={"query": query, "language": language},
        headers=_headers(),
    )
    if error:
        return [], error

    results = data.get("results", []) if isinstance(data, dict) else []
    _CACHE[cache_key] = (time.time(), results)
    return results, None


def genres(language="zh-TW"):
    """類型 id 對照表。回傳 ({id: 名稱}, error)。"""
    data, error = fetch_json(
        GENRE_URL,
        params={"language": language},
        headers=_headers(),
    )
    if error:
        return {}, error
    return {
        g["id"]: ZH_TW_GENRES.get(g["id"], g["name"]) for g in data.get("genres", [])
    }, None


# --------------------------------------------------------------------------
# 比對：為什麼不能直接拿 results[0]
# --------------------------------------------------------------------------
def _normalize(text):
    """比對前先把標點、空白、大小寫的差異抹平。"""
    return re.sub(r"[\s　:：,，.。!！?？'\"’\-—－]", "", str(text or "")).lower()


def match_score(query, movie):
    """片名和某筆搜尋結果的相似度，0.0 ~ 1.0。"""
    q = _normalize(query)
    title = _normalize(movie.get("title"))
    original = _normalize(movie.get("original_title"))
    if not q:
        return 0.0
    if q == title or q == original:
        return 1.0
    if q in title or title in q or q in original or original in q:
        return 0.9
    return max(
        difflib.SequenceMatcher(None, q, title).ratio(),
        difflib.SequenceMatcher(None, q, original).ratio(),
    )


def best_match(query, results, threshold=MATCH_THRESHOLD):
    """從搜尋結果裡挑最像的一筆，都不夠像就回 None。

    原始版本直接取 results[0]，但 TMDB 的排序是它自己的相關度，
    不保證第一筆就是你要的那部片。相似度太低時寧可回 None，
    也不要放一部完全無關的電影到畫面上。
    """
    if not results:
        return None
    ranked = sorted(
        results,
        key=lambda m: (match_score(query, m), m.get("popularity", 0)),
        reverse=True,
    )
    top = ranked[0]
    return top if match_score(query, top) >= threshold else None


def lookup(title, language="zh-TW"):
    """查一個片名，回傳 (最佳結果或 None, error)。"""
    results, error = search(title, language=language)
    if error:
        return None, error
    return best_match(title, results), None


# --------------------------------------------------------------------------
# 批次補資料
# --------------------------------------------------------------------------
def enrich(titles, language="zh-TW", workers=8, source=None):
    """把一串片名補成完整的電影資料。

    回傳 ([{"title": 影城片名, "meta": TMDB 資料, "source": 來源}, ...], errors)，
    查不到的片會被略過。

    workers=1 會退回逐筆查詢，用來對比平行查詢的速度差異。
    """
    titles = list(titles)
    if not titles:
        return [], {}

    def one(title):
        return title, lookup(title, language=language)

    if workers and workers > 1:
        with ThreadPoolExecutor(workers) as pool:
            pairs = list(pool.map(one, titles))
    else:
        pairs = [one(t) for t in titles]

    movies = []
    errors = {}
    for title, (meta, error) in pairs:
        if error:
            errors[title] = error
        elif meta:
            movies.append({"title": title, "meta": meta, "source": source})
    return movies, errors


def poster_url(meta, size="w300"):
    """海報網址；沒有海報回傳 None。"""
    path = (meta or {}).get("poster_path")
    return f"{IMAGE_BASE}{size}{path}" if path else None

---
## 7 ・ 序列 vs 平行

`enrich()` 要對每個片名查一次。50 部電影就是 50 次網路來回，
每次大概 0.1~0.3 秒 —— 逐筆做會等很久。

這些查詢彼此獨立，可以同時發出去。比較看看差多少：

In [ ]:
import time
from movieapp import sources, tmdb

titles, _ = sources.showtimes()
sample = titles[:12]        # 取 12 部來比較，不然序列版要等太久

tmdb.clear_cache()
t0 = time.time()
serial, _ = tmdb.enrich(sample, workers=1)      # 逐筆
serial_time = time.time() - t0

tmdb.clear_cache()
t0 = time.time()
parallel, _ = tmdb.enrich(sample, workers=8)    # 8 條同時
parallel_time = time.time() - t0

print(f"  逐筆查詢  {len(sample)} 部：{serial_time:5.2f} 秒")
print(f"  平行查詢  {len(sample)} 部：{parallel_time:5.2f} 秒")
if parallel_time > 0:
    print(f"  快了約 {serial_time / parallel_time:.1f} 倍")

**平行不是免費的。** 執行緒開太多會被對方限流甚至封鎖，
`workers=8` 是個對外部 API 還算禮貌的數字。

另外原本的網頁版是**逐筆**查詢的，這也是它載入很慢的原因 ——
第 5 章接成服務時，服務端會直接用這裡的 `enrich()`，
所以那個慢的問題到那時候就一併解決了。

---
## 8 ・ 把整份片單補完

把兩家影城的片名全部補上 TMDB 資料。注意保留 `source`，
第 3 章要用它記錄「這部片在哪家上映」。

In [ ]:
by_source, errors = sources.titles_by_source()
genre_names, _ = tmdb.genres()

enriched = {}
for key, items in by_source.items():
    movies, errs = tmdb.enrich(items, source=key, workers=8)
    enriched[key] = movies
    label = sources.SOURCES[key][0]
    rate = len(movies) / len(items) * 100 if items else 0
    print(f"  {label}：{len(items)} 個片名 -> 命中 {len(movies)} 部（{rate:.0f}%）")
    missed = set(items) - {m["title"] for m in movies}
    if missed:
        print(f"     查不到：{sorted(missed)}")

命中率大約九成五。查不到的通常不是程式的問題，而是**那本來就不是電影** ——
「神秘搶先場」是驚喜場、「LIVE VIEWING」是演唱會直播，
電影資料庫裡本來就沒有。

這種時候回 `None` 讓它被略過，比硬塞一部無關的電影正確。

In [ ]:
from movieapp.config import pad

sample = enriched["showtimes"][:10]
print(pad("影城片名", 26) + pad("TMDB 片名", 26) + pad("上映", 12) + pad("評分", 7, "right"))
print("-" * 72)
for m in sample:
    meta = m["meta"]
    print(pad(m["title"][:24], 26) + pad(str(meta.get("title"))[:24], 26)
          + pad(meta.get("release_date") or "-", 12)
          + pad(f"{meta.get('vote_average', 0):.1f}", 7, "right"))

---
## 小結

- 正規 API 的金鑰**放 header 不放網址**，避免從日誌或例外訊息外流
- **`results[0]` 不可信** —— 要自己算相似度挑最佳結果，寧缺勿濫
- 中文片名查中文資料庫，命中率比英文名高很多
- API 給的資料不見得完全符合需求（類型名稱給簡體），自己補要對照穩定的 id
- 速率限制常常是**看 IP** 的，一整間教室共用一個 IP 特別容易撞到，
  所以要有快取
- 獨立的請求可以平行化，但 worker 數量要節制

### 產出

`movieapp/tmdb.py`

### 下一章

**03_資料整合** —— 現在兩家影城各有一份電影清單，很多片重複。
下一章用 TMDB 的電影 id 把它們合併成一份，
再用 pandas 做篩選和排序。